# 01 — Create Dummy Data for Safety Stock Update System

**Purpose:** Generate 5 synthetic Delta tables inside Unity Catalog to power
the Safety Stock Update system (ML scoring, buyer dashboard, approval workflow).

**Catalog / Schema target:**
```
dev.safety_stock_gold
```

**Tables created:**
| Table | Rows | Description |
|---|---|---|
| `materials` | 100 | Material master — plant, category, ABC class, service level |
| `historical_demand` | ~73,000 | Daily demand per material for 2 years (2023–2024) |
| `lead_times` | ~1,200 | PO-level lead times per material-vendor pair |
| `buyers` | 5 | Buyer master with manager mapping and material assignments |
| `current_safety_stock` | 100 | Current SAP safety stock levels per material |

All data is synthetic and follows realistic supply-chain patterns
(seasonal demand, ABC-class variability, lead time distributions).

## 0. Configuration — Edit before running

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
CATALOG = 'dev'                 # Unity Catalog name
SCHEMA  = 'safety_stock_gold'   # Schema / database name
SEED    = 42                    # Random seed for reproducibility
# ─────────────────────────────────────────────────────────────────────────────

print(f'Target: {CATALOG}.{SCHEMA}')
print(f'Random seed: {SEED}')

Target: dev.safety_stock_gold
Random seed: 42


## 1. Create Catalog and Schema (if not already present)

In [ ]:
spark.sql(f'CREATE CATALOG IF NOT EXISTS {CATALOG}')
spark.sql(f'USE CATALOG {CATALOG}')
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA} COMMENT 'Safety Stock Update — synthetic data layer'")
spark.sql(f'USE SCHEMA {SCHEMA}')

print(f'\u2705 Using {CATALOG}.{SCHEMA}')

✅ Using dev.safety_stock_gold


## 2. Imports and Constants

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, BooleanType, TimestampType

rng = np.random.default_rng(SEED)

# ── Data generation constants ─────────────────────────────────────────────────
N_MATERIALS       = 100
PLANTS            = ['P001', 'P002', 'P003']
CATEGORIES        = ['Generators', 'Engines', 'Electrical', 'Controls', 'Accessories']
ABC_CLASSES       = ['A', 'B', 'C']
SERVICE_LEVEL_MAP = {'A': 0.99, 'B': 0.95, 'C': 0.90}
DEMAND_START      = datetime(2023, 1, 1)
DEMAND_END        = datetime(2024, 12, 31)
N_BUYERS          = 5

print('\u2705 Imports complete')

✅ Imports complete


## 3. Helper — pandas DataFrame → Delta Table

Mirrors the `load_csv_to_delta` pattern from the tariff project:
convert a pandas DataFrame to a managed Delta table, register a table-level
comment in Unity Catalog, and print the row count.

In [ ]:
def create_delta_table(pdf: pd.DataFrame, table_name: str, comment: str, overwrite: bool = True) -> pd.DataFrame:
    """
    Convert a pandas DataFrame to a managed Delta table in Unity Catalog.

    Parameters
    ----------
    pdf        : pandas DataFrame to persist
    table_name : target Delta table (unqualified; uses current catalog + schema)
    comment    : table description registered in Unity Catalog
    overwrite  : drop-and-recreate if True (default); append otherwise
    """
    sdf  = spark.createDataFrame(pdf)
    mode = 'overwrite' if overwrite else 'append'

    (
        sdf.write
           .format('delta')
           .mode(mode)
           .option('overwriteSchema', 'true')
           .saveAsTable(f'{CATALOG}.{SCHEMA}.{table_name}')
    )

    safe_comment = comment.replace("'", "''")
    spark.sql(f"COMMENT ON TABLE {CATALOG}.{SCHEMA}.{table_name} IS '{safe_comment}'")

    count = spark.table(f'{CATALOG}.{SCHEMA}.{table_name}').count()
    print(f'  \u2705  {CATALOG}.{SCHEMA}.{table_name}  \u2192  {count:,} rows')
    return pdf

## 4. Generate and Write Tables

### Table 1 — `materials`

100 materials spread across 3 plants, 5 product categories,
and 3 ABC classes (20 % A, 30 % B, 50 % C).

In [ ]:
print('Generating materials master...')

material_ids = [f'M{str(i).zfill(4)}' for i in range(1, N_MATERIALS + 1)]
categories   = rng.choice(CATEGORIES, size=N_MATERIALS).tolist()
abc_classes  = rng.choice(ABC_CLASSES, size=N_MATERIALS, p=[0.2, 0.3, 0.5]).tolist()
plants       = rng.choice(PLANTS, size=N_MATERIALS).tolist()

materials_pdf = pd.DataFrame({
    'material_id':          material_ids,
    'material_desc':        [f'{cat} Component {i}' for i, cat in enumerate(categories, 1)],
    'plant':                plants,
    'category':             categories,
    'abc_class':            abc_classes,
    'service_level_target': [SERVICE_LEVEL_MAP[c] for c in abc_classes],
    'unit_of_measure':      rng.choice(['EA', 'PCE', 'KG', 'SET'], size=N_MATERIALS).tolist(),
    'created_at':           pd.Timestamp('2022-01-01'),
})

create_delta_table(
    pdf        = materials_pdf,
    table_name = 'materials',
    comment    = (
        'Master data for all purchased materials. Contains material metadata, '
        'plant assignment, product category, ABC classification (A=high-value, '
        'C=low-value), and target service level used for safety stock calculation.'
    ),
)

Generating materials master...
  ✅  dev.safety_stock_gold.materials  →  100 rows


### Table 2 — `historical_demand`

730 daily demand rows per material → ~73,000 rows total.  
Demand follows a seasonal sine wave + 5 % annual growth trend + Gaussian noise.  
~10 % of days have zero demand (stock-outs / non-production days).

In [ ]:
print('Generating historical demand (~73k rows)...')

dates = pd.date_range(DEMAND_START, DEMAND_END, freq='D')
rows  = []

for _, mat in materials_pdf.iterrows():
    mid, abc = mat['material_id'], mat['abc_class']

    # Base demand level and noise scale per ABC class
    base         = {'A': 50, 'B': 20, 'C': 5}[abc]
    noise_factor = {'A': 0.15, 'B': 0.30, 'C': 0.50}[abc]

    n = len(dates)
    # Seasonal multiplier: summer peak for generators (sine wave)
    seasonal = 1.0 + 0.3 * np.sin(2 * np.pi * (np.arange(n) / 365 - 0.25))
    # Linear growth trend (~5 % per year)
    trend    = 1.0 + 0.05 * (np.arange(n) / 365)
    demand   = (base * seasonal * trend + rng.normal(0, base * noise_factor, n)).clip(0)

    # ~10 % zero-demand days
    demand[rng.random(n) < 0.10] = 0.0

    for date, qty in zip(dates, demand):
        rows.append({
            'material_id':   mid,
            'plant':         mat['plant'],
            'date':          date,
            'demand_qty':    round(float(qty), 2),
            'source_system': 'SAP',
        })

demand_pdf = pd.DataFrame(rows)
demand_pdf['date'] = pd.to_datetime(demand_pdf['date'])

create_delta_table(
    pdf        = demand_pdf,
    table_name = 'historical_demand',
    comment    = (
        'Daily demand history per material-plant for 2023-2024 (730 days x 100 materials). '
        'Includes seasonal patterns (summer peak), annual growth trend, and ~10% '
        'zero-demand days (stock-outs). Source system is SAP. '
        'Primary table for demand feature engineering.'
    ),
)

Generating historical demand (~73k rows)...
  ✅  dev.safety_stock_gold.historical_demand  →  73,000 rows


### Table 3 — `lead_times`

8–20 purchase orders per material, spread over 2023–2024.  
A-class materials have short, tight lead times; C-class have long, variable ones.

In [ ]:
print('Generating lead times...')

total_days = (DEMAND_END - DEMAND_START).days
lt_rows    = []

for _, mat in materials_pdf.iterrows():
    mid, abc = mat['material_id'], mat['abc_class']

    # Lead time distribution per ABC class
    lt_mean = {'A': 7,  'B': 14, 'C': 21}[abc]
    lt_std  = {'A': 2,  'B': 5,  'C': 10}[abc]
    n_pos   = int(rng.integers(8, 20))

    offsets  = sorted((rng.random(n_pos) * total_days).astype(int).tolist())
    po_dates = [DEMAND_START + timedelta(days=int(d)) for d in offsets]
    lts      = rng.normal(lt_mean, lt_std, n_pos).clip(1, 60).astype(int)

    for po_date, lt in zip(po_dates, lts):
        lt_rows.append({
            'material_id':    mid,
            'plant':          mat['plant'],
            'po_date':        pd.Timestamp(po_date),
            'lead_time_days': int(lt),
            'vendor_id':      f'V{rng.integers(100, 999)}',
        })

lt_pdf = pd.DataFrame(lt_rows)

create_delta_table(
    pdf        = lt_pdf,
    table_name = 'lead_times',
    comment    = (
        'PO-level lead times per material-plant-vendor over 2023-2024. '
        'A-class materials have shorter, tighter lead times; C-class have longer, '
        'more variable lead times. Used for lead time feature engineering '
        '(mean, std dev) in the safety stock ML model.'
    ),
)

Generating lead times...
  ✅  dev.safety_stock_gold.lead_times  →  1,218 rows


### Table 4 — `buyers`

5 buyers, each responsible for ~20 materials.  
Buyers 1–3 report to `MGR001`; buyers 4–5 report to `MGR002`.  
Material assignments are stored as a comma-separated list.

In [ ]:
print('Generating buyers...')

buyer_names = ['Alice Johnson', 'Bob Smith', 'Carol White', 'David Lee', 'Emma Davis']
all_mat_ids = materials_pdf['material_id'].tolist()
rng.shuffle(all_mat_ids)
splits      = np.array_split(all_mat_ids, N_BUYERS)

buyers_pdf = pd.DataFrame([
    {
        'buyer_id':    f'B{str(i).zfill(3)}',
        'buyer_name':  name,
        'email':       f"{name.lower().replace(' ', '.')}@generac.com",
        'manager_id':  'MGR001' if i <= 3 else 'MGR002',
        'material_ids': ','.join(split.tolist()),
        'active':      True,
    }
    for i, (name, split) in enumerate(zip(buyer_names, splits), 1)
])

create_delta_table(
    pdf        = buyers_pdf,
    table_name = 'buyers',
    comment    = (
        'Buyer master data. Each buyer manages a subset of materials and reports '
        'to one of two managers (MGR001 or MGR002). material_ids is a '
        'comma-separated list of material_id values. Used in the approval workflow '
        'to route SS change requests to the correct manager.'
    ),
)

Generating buyers...
  ✅  dev.safety_stock_gold.buyers  →  5 rows


### Table 5 — `current_safety_stock`

Baseline SAP safety stock levels per material (last updated January 2024).  
A-class: ~100 units, B-class: ~50, C-class: ~20 (with ±30 % noise).

In [ ]:
print('Generating current safety stock...')

ss_pdf = pd.DataFrame([
    {
        'material_id':    mat['material_id'],
        'plant':          mat['plant'],
        'current_ss':     int(
            rng.normal(
                {'A': 100, 'B': 50, 'C': 20}[mat['abc_class']],
                {'A': 30,  'B': 15, 'C':  6}[mat['abc_class']],
            ).clip(1)
        ),
        'last_updated':   pd.Timestamp('2024-01-01'),
        'last_updated_by': 'SYSTEM',
    }
    for _, mat in materials_pdf.iterrows()
])

create_delta_table(
    pdf        = ss_pdf,
    table_name = 'current_safety_stock',
    comment    = (
        'Current safety stock levels as stored in SAP (baseline as of Jan 2024). '
        'These are the values the ML model compares against to generate its '
        'new SS recommendations. A last bulk-update was performed by SYSTEM '
        'after the 2023 annual planning cycle.'
    ),
)

Generating current safety stock...
  ✅  dev.safety_stock_gold.current_safety_stock  →  100 rows


## 5. Verify — Row Counts and Schema

In [ ]:
tables = [
    'materials',
    'historical_demand',
    'lead_times',
    'buyers',
    'current_safety_stock',
]

print(f"{'Table':<35} {'Row count':>10}")
print('-' * 47)
for t in tables:
    count = spark.table(f'{CATALOG}.{SCHEMA}.{t}').count()
    print(f'{t:<35} {count:>10,}')

Table                              Row count
-------------------------------------------------
materials                                100
historical_demand                     73,000
lead_times                             1,218
buyers                                     5
current_safety_stock                     100


### Sample rows from `materials`

In [ ]:
display(
    spark.table(f'{CATALOG}.{SCHEMA}.materials').limit(10)
)

### Sample rows from `historical_demand`

In [ ]:
display(
    spark.table(f'{CATALOG}.{SCHEMA}.historical_demand')
        .orderBy('material_id', 'date')
        .limit(10)
)

### Sample rows from `current_safety_stock`

In [ ]:
display(
    spark.table(f'{CATALOG}.{SCHEMA}.current_safety_stock')
        .join(
            spark.table(f'{CATALOG}.{SCHEMA}.materials').select('material_id', 'abc_class', 'category'),
            on='material_id'
        )
        .orderBy('abc_class', 'material_id')
        .limit(10)
)

## 6. Add Column-Level Comments (helps Genie understand fields)

These comments appear in Unity Catalog and are read by Genie to improve
SQL generation accuracy — same pattern used in the tariff simulator project.

In [ ]:
col_comments = {
    'materials': {
        'material_id':           'Unique material identifier (e.g. M0001). Primary key. Also called part number.',
        'material_desc':         'Human-readable material description.',
        'plant':                 'Manufacturing plant code (P001, P002, P003).',
        'category':              'Product category: Generators, Engines, Electrical, Controls, or Accessories.',
        'abc_class':             'ABC classification: A = high-value/high-velocity, B = medium, C = low-value/slow-moving.',
        'service_level_target':  'Target fill-rate service level (0.90, 0.95, or 0.99). Drives the Z-score in the safety stock formula.',
        'unit_of_measure':       'Unit of measure: EA (each), PCE (piece), KG (kilogram), or SET.',
        'created_at':            'Date the material record was created in SAP.',
    },
    'historical_demand': {
        'material_id':   'Material identifier. Foreign key to materials.material_id.',
        'plant':         'Plant code. Foreign key to materials.plant.',
        'date':          'Calendar date of the demand record (daily granularity).',
        'demand_qty':    'Units demanded on this date. Zero on stock-out or non-production days.',
        'source_system': 'Source ERP system (SAP).',
    },
    'lead_times': {
        'material_id':    'Material identifier. Foreign key to materials.material_id.',
        'plant':          'Plant code.',
        'po_date':        'Date the purchase order was placed with the vendor.',
        'lead_time_days': 'Actual lead time in calendar days from PO placement to goods receipt.',
        'vendor_id':      'Vendor identifier (e.g. V123). Links to vendor records.',
    },
    'buyers': {
        'buyer_id':     'Unique buyer identifier (e.g. B001). Primary key.',
        'buyer_name':   'Full name of the procurement buyer.',
        'email':        'Corporate email address.',
        'manager_id':   'Manager identifier (MGR001 or MGR002). Approval requests are routed to this manager.',
        'material_ids': 'Comma-separated list of material_id values managed by this buyer.',
        'active':       'True if the buyer account is active.',
    },
    'current_safety_stock': {
        'material_id':    'Material identifier. Foreign key to materials.material_id.',
        'plant':          'Plant code.',
        'current_ss':     'Current safety stock level in SAP (units). This is the baseline the ML model compares against.',
        'last_updated':   'Date this safety stock value was last changed in SAP.',
        'last_updated_by': 'User or process that last updated this record.',
    },
}

for table, cols in col_comments.items():
    for col, comment in cols.items():
        safe_comment = comment.replace("'", "''")
        spark.sql(
            f'ALTER TABLE {CATALOG}.{SCHEMA}.{table} '
            f"ALTER COLUMN {col} COMMENT '{safe_comment}'"
        )
    print(f'  \u2705  Column comments added \u2192 {table}')

print('\n\u2705 All column-level comments applied.')

  ✅  Column comments added → materials
  ✅  Column comments added → historical_demand
  ✅  Column comments added → lead_times
  ✅  Column comments added → buyers
  ✅  Column comments added → current_safety_stock

✅ All column-level comments applied.


## Next Step

Run **`02_medallion_pipeline`** to transform the bronze tables into silver
(cleaned weekly demand, outlier-free lead times) and gold
(ML-ready feature table `safety_stock_features`) layers.

Then run **`03_train_model`** to train the RandomForest safety stock model
and register it in MLflow.